# Model Building and Training — Fraud Detection

**Project:** Improved Detection of Fraud Cases — Adey Innovations Inc.
**Task:** Task 2 — Model Building and Training
**Datasets:** `fraud_data_clean.csv` | `creditcard.csv` (processed splits)

---

## Objectives

1. Load preprocessed train/test splits for both datasets.
2. Train a Logistic Regression baseline model.
3. Train a Random Forest ensemble model with hyperparameter tuning.
4. Evaluate both models using AUC-PR, F1-Score, and Confusion Matrix.
5. Apply Stratified K-Fold cross-validation (k=5).
6. Compare models side-by-side and select the best with justification.

---

## Notebook Structure

| Section | Description |
|---------|-------------|
| 1 | Imports & Configuration |
| 2 | Load Processed Splits |
| 3 | Baseline — Logistic Regression |
| 4 | Ensemble — Random Forest |
| 5 | Cross-Validation |
| 6 | Model Comparison & Selection |
| 7 | Save Models |

In [3]:
import logging
import warnings
import os

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    f1_score, confusion_matrix, ConfusionMatrixDisplay,
    precision_recall_curve, average_precision_score,
    classification_report
)
from sklearn.model_selection import (
    StratifiedKFold, cross_val_score, train_test_split
)

warnings.filterwarnings('ignore')

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s — %(levelname)s — %(message)s'
)
logger = logging.getLogger(__name__)

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120

PROCESSED_DIR = '../data/processed/'
MODELS_DIR    = '../models/'
os.makedirs(MODELS_DIR, exist_ok=True)

logger.info('Libraries loaded successfully.')

2026-06-13 19:05:33,725 — INFO — Libraries loaded successfully.


## 2. Load Processed Splits

Load the SMOTE-resampled training splits and held-out test splits produced
by the EDA and feature engineering notebooks. Both datasets are loaded so
models are trained and evaluated in parallel.

In [4]:
try:
    # --- Fraud_Data splits ---
    fraud_X_train = pd.read_csv(PROCESSED_DIR + 'fraud_X_train.csv')
    fraud_X_test  = pd.read_csv(PROCESSED_DIR + 'fraud_X_test.csv')
    fraud_y_train = pd.read_csv(PROCESSED_DIR + 'fraud_y_train.csv').squeeze()
    fraud_y_test  = pd.read_csv(PROCESSED_DIR + 'fraud_y_test.csv').squeeze()

    logger.info(f'Fraud — Train: {fraud_X_train.shape} | Test: {fraud_X_test.shape}')
    logger.info(f'Fraud — Train class dist: {fraud_y_train.value_counts().to_dict()}')

except FileNotFoundError as e:
    logger.error(f'Fraud split not found: {e}')
    raise
except Exception as e:
    logger.error(f'Unexpected error loading Fraud splits: {e}')
    raise

try:
    # --- CreditCard splits ---
    cc_X_train = pd.read_csv(PROCESSED_DIR + 'cc_X_train.csv')
    cc_X_test  = pd.read_csv(PROCESSED_DIR + 'cc_X_test.csv')
    cc_y_train = pd.read_csv(PROCESSED_DIR + 'cc_y_train.csv').squeeze()
    cc_y_test  = pd.read_csv(PROCESSED_DIR + 'cc_y_test.csv').squeeze()

    logger.info(f'CreditCard — Train: {cc_X_train.shape} | Test: {cc_X_test.shape}')
    logger.info(f'CreditCard — Train class dist: {cc_y_train.value_counts().to_dict()}')

except FileNotFoundError as e:
    logger.error(f'CreditCard split not found: {e}')
    raise
except Exception as e:
    logger.error(f'Unexpected error loading CreditCard splits: {e}')
    raise

2026-06-13 19:06:38,404 — INFO — Fraud — Train: (219136, 12) | Test: (30223, 12)
2026-06-13 19:06:38,412 — INFO — Fraud — Train class dist: {0: 109568, 1: 109568}
2026-06-13 19:06:45,237 — INFO — CreditCard — Train: (453204, 30) | Test: (56746, 30)
2026-06-13 19:06:45,244 — INFO — CreditCard — Train class dist: {0: 226602, 1: 226602}
